In [ ]:
import os
import getpass
from langchain.chat_models import init_chat_model

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key in environment vars ")

os.environ["LANGSMITH_TRACING"] = "true"

### Human In the Loop MiddleWare

Pause agent execution for human approval, editing, or rejection of tool calls before they execute. Human-in-the-loop is useful for the following:
- High-stakes operations requiring human approval (e.g. database writes, financial transactions).
- Compliance workflows where human oversight is mandatory.
- Long-running conversations where human feedback guides the agent.

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime

# Creating Tools
@tool
def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}' and body '{body}'"

/home/brijeshdhaker/IdeaProjects/bd-crewai-module/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
import os
#
agent=create_agent(
    model=f"openai:{os.environ["OPENAI_MODEL_NAME"]}",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool":False,
            }
        )
    ]
)

##### Test Human Action : Approve Action

In [3]:
from langchain_core.messages import HumanMessage

approve_config = {"configurable": {"thread_id": "test-approve"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'")]},
    config=approve_config
)

#
result

{'messages': [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='c98721e0-712e-4766-a3c3-11ac89ec5d29'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 166, 'total_tokens': 285, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma4:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-165', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e21cc-2959-7a42-8642-4b4ea68c40d8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you ?', 'recipient': 'brijeshdhaker@gmail.com', 'subject': 'Hello'}, 'id': 'call_6fpp2zpj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 119, 'total_tokens': 285, 'input_token_details': {}, 'output_token_details': {}

In [4]:
from langgraph.types import Command

# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
#
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=approve_config
)

print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: Lbs an email successfully sent.


In [5]:
result

{'messages': [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='c98721e0-712e-4766-a3c3-11ac89ec5d29'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 166, 'total_tokens': 285, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma4:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-165', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e21cc-2959-7a42-8642-4b4ea68c40d8-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you ?', 'recipient': 'brijeshdhaker@gmail.com', 'subject': 'Hello'}, 'id': 'call_6fpp2zpj', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 119, 'total_tokens': 285, 'input_token_details': {}, 'output_token_details': {}

##### Test Human Action : Reject Action

In [6]:
reject_config = {"configurable": {"thread_id": "test-reject"}}
# Step 1: Request
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'")]},
    config=reject_config
)

In [7]:
# Step 2: Reject
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=reject_config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: ### Awaiting Confirmation

I have attempted to send the email to 'brijeshdhaker@gmail.com' with the subject 'Hello' and body 'How are you ?'.

However, your system rejected the tool call.

Would you like me to try again, or would you like to cancel the action?


In [8]:
result

{'messages': [HumanMessage(content="Send email to 'brijeshdhaker@gmail.com' with subject 'Hello' and body 'How are you ?'", additional_kwargs={}, response_metadata={}, id='5a151998-dd22-435a-a726-b3c893016200'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 264, 'prompt_tokens': 166, 'total_tokens': 430, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma4:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-698', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e21cc-f479-7633-a3f8-cdaf2adf5a0a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'How are you ?', 'recipient': 'brijeshdhaker@gmail.com', 'subject': 'Hello'}, 'id': 'call_1q9uam94', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 264, 'total_tokens': 430, 'input_token_details': {}, 'output_token_details': {}

##### Test Human Action : Editing Action 

In [9]:
edit_config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'")]},
    config=edit_config
)

In [10]:
result

{'messages': [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='b388a9da-b229-4251-a472-101ed88f42dd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 235, 'prompt_tokens': 159, 'total_tokens': 394, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma4:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-282', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e21cd-57e5-7581-bcca-9a1fccbcfe54-0', tool_calls=[{'name': 'send_email_tool', 'args': {'body': 'Hello', 'recipient': 'wrong@email.com', 'subject': 'Test'}, 'id': 'call_kwizd75j', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 235, 'total_tokens': 394, 'input_token_details': {}, 'output_token_details': {}})],
 '__interrupt__': [Interrupt(

In [11]:
# Step 2: Edit and approve
if "__interrupt__" in result:
    print("⏸️ Paused! Editing...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",      # Tool name
                            "args": {                   # New arguments
                                "recipient": "neetadhk@gmail.com",
                                "subject": "Editated Subject",
                                "body": "This was edited by human before sending."
                            }
                        }
                    }
                ]
            }
        ),
        config=edit_config
    )
    
    print(f"✏️ Result: {result['messages'][-1].content}")

⏸️ Paused! Editing...
✏️ Result: 


In [12]:
result

{'messages': [HumanMessage(content="Send email to 'wrong@email.com' with subject 'Test' and body 'Hello'", additional_kwargs={}, response_metadata={}, id='b388a9da-b229-4251-a472-101ed88f42dd'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 235, 'prompt_tokens': 159, 'total_tokens': 394, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'gemma4:latest', 'system_fingerprint': 'fp_ollama', 'id': 'chatcmpl-282', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e21cd-57e5-7581-bcca-9a1fccbcfe54-0', tool_calls=[{'type': 'tool_call', 'name': 'send_email_tool', 'args': {'recipient': 'neetadhk@gmail.com', 'subject': 'Editated Subject', 'body': 'This was edited by human before sending.'}, 'id': 'call_kwizd75j'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 159, 'output_tokens': 235, 'total_tokens': 394, 'input_token_details': {}, 'output_to